# OpenRouter LLM + Perseus MCP Tool Interaction

This optional notebook demonstrates an LLM-driven tool loop. It sends the local `perseus` MCP tool schemas to OpenRouter, lets the selected model request tools, executes those requests through FastMCP, and sends the results back to the model.

The LLM does not call MCP tools directly. This notebook is the client-side bridge between OpenRouter tool calls and the local MCP server.

> Requirements: install project dependencies, have internet access to OpenRouter and Perseus/Scaife, and provide an OpenRouter API key. The default model is free, but OpenRouter rate limits and availability apply. Selecting a different model may incur charges.

> This is a client-side LLM adapter, not part of the MCP server runtime. The MCP server itself does not require an API key.

## Configuration

Copy `.env.example` to `.env` in the project root and replace the placeholder with an OpenRouter key:

```dotenv
OPENROUTER_API_KEY=sk-or-v1-...
```

Get your API key at [openrouter.ai](https://openrouter.ai/settings/keys). See [OpenRouter's API key documentation](https://openrouter.ai/docs/api-keys) for authentication details. You can also set `OPENROUTER_API_KEY` in your environment or enter it securely when prompted. The notebook reads the key at runtime and does not print it. Clear notebook outputs before committing after any credentialed run.

Set `OPENROUTER_MODEL` to choose a different tool-calling model. The default is the free [NVIDIA: Nemotron 3 Super](https://openrouter.ai/nvidia/nemotron-3-super-120b-a12b:free) model: `nvidia/nemotron-3-super-120b-a12b:free`.

OpenRouter documentation:

- [Quickstart](https://openrouter.ai/docs/quickstart)
- [Tool calling](https://openrouter.ai/docs/guides/features/tool-calling)

In [1]:
from pathlib import Path
from getpass import getpass
import importlib
import json
import os
import sys

import httpx
from dotenv import load_dotenv
from IPython.display import Markdown, display

# Make `import server` work when the notebook is opened from examples/.
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "server.py").exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))
load_dotenv(REPO_ROOT / ".env", override=False)

from fastmcp import Client
import server

# Reload local edits when this notebook is rerun in an existing kernel.
server = importlib.reload(server)
mcp = server.mcp

OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"
OPENROUTER_MODEL = os.getenv(
    "OPENROUTER_MODEL", "nvidia/nemotron-3-super-120b-a12b:free"
)
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY") or getpass("OpenRouter API key: ")

if not OPENROUTER_API_KEY:
    raise RuntimeError("OPENROUTER_API_KEY is required.")

print(f"Model: {OPENROUTER_MODEL}")

Model: nvidia/nemotron-3-super-120b-a12b:free


## Convert MCP tools to OpenRouter function tools

OpenRouter uses the OpenAI-compatible function-calling format. Each MCP tool already has a name, description, and JSON input schema, so the conversion is mechanical.

In [2]:
async with Client(mcp) as client:
    mcp_tools = await client.list_tools()

openrouter_tools = [
    {
        "type": "function",
        "function": {
            "name": tool.name,
            "description": tool.description or "",
            "parameters": tool.inputSchema,
        },
    }
    for tool in mcp_tools
]

print(f"Exposing {len(openrouter_tools)} MCP tools to the LLM:")
print(", ".join(tool["function"]["name"] for tool in openrouter_tools))

Exposing 12 MCP tools to the LLM:
get_passage, get_passage_plus, get_passage_plaintext, get_valid_references, get_capabilities, list_text_groups, get_author_resources, get_work_resources, get_label, get_first_urn, get_prev_next_urn, search_perseus


## LLM and MCP tool loop

The loop below:

1. sends the conversation and tool schemas to OpenRouter;
2. appends the assistant response to the conversation;
3. executes any requested tools through `Client(mcp)`;
4. appends tool results using the matching `tool_call_id`;
5. repeats until the model returns a final answer or the round limit is reached.

In [3]:
def tool_text(result):
    """Extract text content blocks from a FastMCP CallToolResult."""
    return "\n".join(
        block.text for block in result.content if getattr(block, "text", None) is not None
    )


def openrouter_completion(messages):
    response = httpx.post(
        OPENROUTER_URL,
        headers={
            "Authorization": f"Bearer {OPENROUTER_API_KEY}",
            "Content-Type": "application/json",
        },
        json={
            "model": OPENROUTER_MODEL,
            "messages": messages,
            "tools": openrouter_tools,
            "tool_choice": "auto",
        },
        timeout=120.0,
    )
    try:
        response.raise_for_status()
    except httpx.HTTPStatusError as exc:
        raise RuntimeError(
            f"OpenRouter error {response.status_code}: {response.text[:2000]}"
        ) from exc
    return response.json()


async def run_llm_with_mcp(user_prompt, max_rounds=8):
    messages = [
        {
            "role": "system",
            "content": (
                "You are a careful Ancient Greek research assistant. Use the Perseus MCP tools "
                "for factual text retrieval. Discover CTS edition URNs before fetching passages. "
                "Prefer specific discovery tools over the full capabilities inventory."
            ),
        },
        {"role": "user", "content": user_prompt},
    ]

    async with Client(mcp) as client:
        for round_number in range(1, max_rounds + 1):
            completion = openrouter_completion(messages)
            message = completion["choices"][0]["message"]
            assistant_message = {"role": "assistant", "content": message.get("content")}
            if message.get("tool_calls"):
                assistant_message["tool_calls"] = message["tool_calls"]
            messages.append(assistant_message)

            display(Markdown(f"### LLM round {round_number}"))
            if message.get("content"):
                display(Markdown(message["content"]))

            tool_calls = message.get("tool_calls") or []
            if not tool_calls:
                return message.get("content", ""), messages

            for tool_call in tool_calls:
                tool_name = tool_call["function"]["name"]
                tool_args = json.loads(tool_call["function"].get("arguments") or "{}")
                display(Markdown(f"**Tool request:** `{tool_name}`\n\n```json\n{json.dumps(tool_args, ensure_ascii=False, indent=2)}\n```"))

                try:
                    result = await client.call_tool(tool_name, tool_args)
                    result_text = tool_text(result)
                except Exception as exc:
                    result_text = f"Tool error: {type(exc).__name__}: {exc}"

                max_tool_result_chars = 20_000
                tool_content = result_text[:max_tool_result_chars]
                if len(result_text) > max_tool_result_chars:
                    tool_content += "\n...[tool result truncated]"

                display(Markdown(f"**Tool result:** `{tool_name}`\n\n```text\n{tool_content[:4000]}\n```"))
                messages.append(
                    {
                        "role": "tool",
                        "tool_call_id": tool_call["id"],
                        "content": tool_content,
                    }
                )

    raise RuntimeError(f"The model did not produce a final answer within {max_rounds} rounds.")

## Run an example research request

This prompt is designed to require discovery and passage retrieval rather than relying on the model's memory.

In [4]:
prompt = (
    "Use the Perseus MCP tools to identify the Greek Iliad edition currently available from "
    "Perseus CTS, retrieve Iliad 1.1, and briefly explain what text was returned. "
    "Include the exact CTS URN you used."
)

final_answer, conversation = await run_llm_with_mcp(prompt)
display(Markdown("## Final answer"))
display(Markdown(final_answer))

### LLM round 1

**Tool request:** `get_author_resources`

```json
{
  "author": "Homer",
  "language": "greek"
}
```

**Tool result:** `get_author_resources`

```text
{
  "query": "Homer",
  "language": "grc",
  "match_count": 2,
  "authors": [
    {
      "urn": "urn:cts:greekLit:tlg0012",
      "names": [
        "Homer"
      ],
      "works_count": 2,
      "works": [
        {
          "urn": "urn:cts:greekLit:tlg0012.tlg001",
          "language": "grc",
          "titles": [
            "Iliad"
          ],
          "editions": [
            {
              "type": "edition",
              "urn": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc1",
              "label": "Iliad",
              "description": "Perseus:bib:oclc,29448041, Homer. Homeri Opera in five volumes. Oxford, Oxford University Press. 1920."
            }
          ],
          "translations": [
            {
              "type": "translation",
              "urn": "urn:cts:greekLit:tlg0012.tlg001.perseus-eng1",
              "language": "eng",
              "label": "Iliad",
              "description": "Perseus:bib:oclc,38101377, Perseus:bib:isbn,0674991885, Perseus:bib:isbn,0674991893, Homer. The Iliad with an English Translation by A.T. Murray, Ph.D. in two volumes. Cambridge, MA., Harvard University Press; London, William Heinemann, Ltd. 1924."
            },
            {
              "type": "translation",
              "urn": "urn:cts:greekLit:tlg0012.tlg001.perseus-eng2",
              "language": "eng",
              "label": "The Iliad",
              "description": "Homer. The Iliad of Homer. Rendered into English prose for the use of those who cannot read the original. Samuel Butler. Longmans, Green and Co. 39 Paternoster Row, London. New York and Bombay. 1898 (?)."
            }
          ],
          "resources": [
            {
              "type": "edition",
              "urn": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc1",
              "label": "Iliad",
              "description": "Perseus:bib:oclc,29448041, Homer. Homeri Opera in five volumes. Oxford, Oxford University Press. 1920."
            },
            {
              "type": "translation",
              "urn": "urn:cts:greekLit:tlg0012.tlg001.perseus-eng1",
              "language": "eng",
              "label": "Iliad",
              "description": "Perseus:bib:oclc,38101377, Perseus:bib:isbn,0674991885, Perseus:bib:isbn,0674991893, Homer. The Iliad with an English Translation by A.T. Murray, Ph.D. in two volumes. Cambridge, MA., Harvard University Press; London, William Heinemann, Ltd. 1924."
            },
            {
              "type": "translation",
              "urn": "urn:cts:greekLit:tlg0012.tlg001.perseus-eng2",
              "language": "eng",
              "label": "The Iliad",
              "description": "Homer. The Iliad of Homer. Rendered into English prose for the use of those who cannot read the original. Samuel Butler. Longmans, Green and Co. 39 Paternoster Row, London. New York and Bombay. 1898 (?)."
            }
          ]
        },
        {
          "urn": "urn:cts:greekLit:tlg0012.tlg002",
          "language": "grc",
          "titles": [
            "Odyssey"
          ],
          "editions": [
            {
              "type": "edition",
              "urn": "urn:cts:greekLit:tlg0012.tlg002.perseus-grc1",
              "label": "Odyssey",
              "description": "Perseus:bib:oclc,22584673, Perseus:bib:isbn,0674995619, Perseus:bib:isbn,0674995627, Homer. The Odyssey with an English Translation by A.T. Murray, PH.D. in two volumes. Cambridge, MA., Harvard University Press; London, William Heinemann, Ltd. 1919."
            }
          ],
          "translations": [
            {
              "type": "translation",
              "urn": "urn:cts:greekLit:tlg0012.tlg002.perseus-eng1",
              "language": "eng",
              "label": "Odyssey",
              "description": "Perseus:bib:oclc,22584673, Perseus:bib:isbn,0674995619, Perseus:bib:isbn,0674995627, Homer. The Odyssey with an English Translation by A.T. Murray, PH.D. in two volumes. Cambridge, MA., Harvard University Press; London
```

### LLM round 2

**Tool request:** `get_passage_plaintext`

```json
{
  "urn": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc1:1.1"
}
```

**Tool result:** `get_passage_plaintext`

```text
μῆνιν ἄειδε θεὰ Πηληϊάδεω Ἀχιλῆος
```

### LLM round 3

**Greek Iliad edition identified:**  
`urn:cts:greekLit:tlg0012.tlg001.perseus-grc1`  
This is the Perseus Greek edition of Homer’s *Iliad* (based on the Oxford Classical Text, 1920).

**Passage retrieved (Iliad 1.1):**  
`urn:cts:greekLit:tlg0012.tlg001.perseus-grc1:1.1`

**Returned text:**  
`μῆνιν ἄειδε θεὰ Πηληϊάδεω Ἀχιλῆος`

**Explanation:**  
The passage is the famous opening line of the *Iliad*: “Sing, goddess, the anger of Peleus’ son Achilles.” In Greek, it reads *μῆνιν ἄειδε θεὰ Πηληϊάδεω Ἀχιλῆος*, invoking the Muse to tell the story of Achilles’ wrath that drives the epic. The Perseus CTS service returned this line in its original Greek characters, confirming that the edition is accessible and correctly cited.

## Final answer

**Greek Iliad edition identified:**  
`urn:cts:greekLit:tlg0012.tlg001.perseus-grc1`  
This is the Perseus Greek edition of Homer’s *Iliad* (based on the Oxford Classical Text, 1920).

**Passage retrieved (Iliad 1.1):**  
`urn:cts:greekLit:tlg0012.tlg001.perseus-grc1:1.1`

**Returned text:**  
`μῆνιν ἄειδε θεὰ Πηληϊάδεω Ἀχιλῆος`

**Explanation:**  
The passage is the famous opening line of the *Iliad*: “Sing, goddess, the anger of Peleus’ son Achilles.” In Greek, it reads *μῆνιν ἄειδε θεὰ Πηληϊάδεω Ἀχιλῆος*, invoking the Muse to tell the story of Achilles’ wrath that drives the epic. The Perseus CTS service returned this line in its original Greek characters, confirming that the edition is accessible and correctly cited.

## Inspect the complete message history

The conversation contains the original prompt, assistant tool requests, tool results, and the final answer.

In [5]:
print(json.dumps(conversation, ensure_ascii=False, indent=2))

[
  {
    "role": "system",
    "content": "You are a careful Ancient Greek research assistant. Use the Perseus MCP tools for factual text retrieval. Discover CTS edition URNs before fetching passages. Prefer specific discovery tools over the full capabilities inventory."
  },
  {
    "role": "user",
    "content": "Use the Perseus MCP tools to identify the Greek Iliad edition currently available from Perseus CTS, retrieve Iliad 1.1, and briefly explain what text was returned. Include the exact CTS URN you used."
  },
  {
    "role": "assistant",
    "content": null,
    "tool_calls": [
      {
        "type": "function",
        "index": 0,
        "id": "chatcmpl-tool-b647a75c041f3508",
        "function": {
          "name": "get_author_resources",
          "arguments": "{\"author\": \"Homer\", \"language\": \"greek\"}"
        }
      }
    ]
  },
  {
    "role": "tool",
    "tool_call_id": "chatcmpl-tool-b647a75c041f3508",
    "content": "{\n  \"query\": \"Homer\",\n  \"language\

# Notebook version

<div style="float: left;">
  <table>
    <tr>
      <td><strong>Author</strong></td>
      <td>Tony Jurg</td>
    </tr>
    <tr>
      <td><strong>Version</strong></td>
      <td>1.0</td>
    </tr>
    <tr>
      <td><strong>Date</strong></td>
      <td>June 3, 2026</td>
    </tr>
  </table>
</div>